# WBCBench 2026 — Aşama 10A: ResNet-50 Final Training — Seed 42

Final eğitim protokolü:

- Model: **ResNet-50**
- Pretraining: ImageNet
- Seed: **42**
- Train: `phase2_train`
- Validation: `phase2_eval`
- Test: **KULLANILMAZ**
- Primary selection metric: validation Macro-F1
- Max epoch: 12
- Early stopping patience: 3
- Batch: 8
- AdamW, LR=1e-4, weight decay=1e-4
- Best **tam model checkpoint'i** kaydedilir.

Kernel: **medxai_gpu**.


In [1]:
from pathlib import Path
import random,time,json,csv
import numpy as np, torch, torchvision
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets,transforms,models
from sklearn.metrics import f1_score,balanced_accuracy_score,accuracy_score

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type=="cuda"
print("Torch:",torch.__version__)
print("Torchvision:",torchvision.__version__)
print("GPU:",torch.cuda.get_device_name(0))


Torch: 2.11.0+cu126
Torchvision: 0.26.0+cu126
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
ROOT=Path.cwd()
TRAIN=ROOT/"wbc_project"/"data"/"processed"/"phase2_train"
VAL=ROOT/"wbc_project"/"data"/"processed"/"phase2_eval"
OUT=ROOT/"outputs"/"final"/"resnet50"/"seed42"
OUT.mkdir(parents=True,exist_ok=True)
assert TRAIN.exists() and VAL.exists()
print("Output:",OUT.resolve())
print("TEST split bu notebook'ta tanımlanmıyor.")


Output: C:\Users\sarah\Desktop\scı5\outputs\final\resnet50\seed42
TEST split bu notebook'ta tanımlanmıyor.


In [3]:
train_tf=transforms.Compose([
 transforms.Resize((224,224)),transforms.RandomHorizontalFlip(.5),
 transforms.RandomRotation(10),transforms.ToTensor(),
 transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
val_tf=transforms.Compose([
 transforms.Resize((224,224)),transforms.ToTensor(),
 transforms.Normalize([.485,.456,.406],[.229,.224,.225])])

train_ds=datasets.ImageFolder(TRAIN,transform=train_tf)
val_ds=datasets.ImageFolder(VAL,transform=val_tf)
assert train_ds.classes==val_ds.classes and len(train_ds.classes)==13

g=torch.Generator(); g.manual_seed(SEED)
BATCH=8
train_loader=DataLoader(train_ds,batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True,generator=g)
val_loader=DataLoader(val_ds,batch_size=BATCH,shuffle=False,num_workers=0,pin_memory=True)

print("Classes:",train_ds.classes)
print("Train:",len(train_ds),"Validation:",len(val_ds),"Batch:",BATCH)


Classes: ['BA', 'BL', 'BNE', 'EO', 'LY', 'MMY', 'MO', 'MY', 'PC', 'PLY', 'PMY', 'SNE', 'VLY']
Train: 49794 Validation: 10700 Batch: 8


In [4]:
model=models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc=nn.Linear(model.fc.in_features,13)
model=model.to(device)

criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=1e-4)
scaler=torch.amp.GradScaler("cuda")

print("Parameters:",f"{sum(p.numel() for p in model.parameters()):,}")


Parameters: 23,534,669


In [5]:
def train_epoch():
    model.train(); total=0.; n=0
    for i,(x,y) in enumerate(train_loader,1):
        x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda",dtype=torch.float16):
            z=model(x); loss=criterion(z,y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        total+=loss.item()*y.size(0); n+=y.size(0)
        if i%1000==0:
            print(f"  batch {i}/{len(train_loader)} | train_loss={total/n:.4f}")
    return total/n

@torch.inference_mode()
def validate():
    model.eval(); total=0.; n=0; ys=[]; ps=[]
    for x,y in val_loader:
        x=x.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
        with torch.autocast("cuda",dtype=torch.float16):
            z=model(x); loss=criterion(z,y)
        total+=loss.item()*y.size(0); n+=y.size(0)
        ys+=y.cpu().tolist(); ps+=z.argmax(1).cpu().tolist()
    return {"val_loss":total/n,
            "macro_f1":f1_score(ys,ps,average="macro",zero_division=0),
            "balanced_acc":balanced_accuracy_score(ys,ps),
            "accuracy":accuracy_score(ys,ps)}


In [6]:
MAX_EPOCHS=12; PATIENCE=3
best=-1.; bad=0; history=[]
best_path=OUT/"resnet50_seed42_best.pt"
config={"model":"resnet50","pretraining":"ImageNet1K_V2","seed":SEED,
        "max_epochs":MAX_EPOCHS,"patience":PATIENCE,"batch_size":BATCH,
        "lr":1e-4,"weight_decay":1e-4,"input_size":224,
        "selection_metric":"validation_macro_f1","test_used":False,
        "classes":train_ds.classes}
(OUT/"config.json").write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding="utf-8")

for epoch in range(1,MAX_EPOCHS+1):
    print("\n"+"="*72)
    print(f"RESNET-50 FINAL — SEED 42 — EPOCH {epoch}/{MAX_EPOCHS}")
    print("="*72)
    t=time.time(); tl=train_epoch(); m=validate()
    row={"epoch":epoch,"train_loss":tl,**m,"minutes":(time.time()-t)/60}
    history.append(row)
    print(f"train_loss={tl:.4f} | val_loss={m['val_loss']:.4f} | "
          f"Macro-F1={m['macro_f1']:.4f} | Balanced Acc={m['balanced_acc']:.4f} | "
          f"Acc={m['accuracy']:.4f} | {row['minutes']:.1f} dk")

    if m["macro_f1"]>best:
        best=m["macro_f1"]; bad=0
        torch.save({"model_state_dict":model.state_dict(),"classes":train_ds.classes,
                    "seed":SEED,"epoch":epoch,"val_macro_f1":best,"config":config},best_path)
        print("✓ Yeni best checkpoint:",best_path)
    else:
        bad+=1; print(f"İyileşme yok: {bad}/{PATIENCE}")

    (OUT/"history.json").write_text(json.dumps(history,ensure_ascii=False,indent=2),encoding="utf-8")
    with open(OUT/"history.csv","w",newline="",encoding="utf-8") as f:
        w=csv.DictWriter(f,fieldnames=history[0].keys()); w.writeheader(); w.writerows(history)

    if bad>=PATIENCE:
        print("✓ Early stopping."); break

result={"seed":SEED,"best_val_macro_f1":best,"epochs_completed":len(history),
        "best_checkpoint":str(best_path)}
(OUT/"result.json").write_text(json.dumps(result,indent=2),encoding="utf-8")

print("\n"+"="*72)
print("RESNET-50 FINAL TRAINING — SEED 42 SONUCU")
print("="*72)
print("Best validation Macro-F1:",f"{best:.4f}")
print("Epochs completed:",len(history))
print("Best checkpoint:",best_path)
print("✓ TEST SPLIT KULLANILMADI.")



RESNET-50 FINAL — SEED 42 — EPOCH 1/12
  batch 1000/6225 | train_loss=0.6125
  batch 2000/6225 | train_loss=0.4913
  batch 3000/6225 | train_loss=0.4364
  batch 4000/6225 | train_loss=0.3982
  batch 5000/6225 | train_loss=0.3724
  batch 6000/6225 | train_loss=0.3537
train_loss=0.3494 | val_loss=0.3043 | Macro-F1=0.5937 | Balanced Acc=0.5623 | Acc=0.9141 | 8.8 dk
✓ Yeni best checkpoint: C:\Users\sarah\Desktop\scı5\outputs\final\resnet50\seed42\resnet50_seed42_best.pt

RESNET-50 FINAL — SEED 42 — EPOCH 2/12
  batch 1000/6225 | train_loss=0.2309
  batch 2000/6225 | train_loss=0.2329
  batch 3000/6225 | train_loss=0.2302
  batch 4000/6225 | train_loss=0.2248
  batch 5000/6225 | train_loss=0.2188
  batch 6000/6225 | train_loss=0.2143
train_loss=0.2138 | val_loss=0.2357 | Macro-F1=0.6492 | Balanced Acc=0.6225 | Acc=0.9263 | 8.7 dk
✓ Yeni best checkpoint: C:\Users\sarah\Desktop\scı5\outputs\final\resnet50\seed42\resnet50_seed42_best.pt

RESNET-50 FINAL — SEED 42 — EPOCH 3/12
  batch 1000/622